# Tool-Calling Research Agent (Teacher)

**Phase 1** of the BBB conference demo pipeline.

This notebook implements a tool-calling agent using the **OpenAI Responses API**
with reasoning support (gpt-5.4). The agent researches a given stock ticker by
calling financial tools (yfinance), then produces a structured research memo.

The full conversation trajectory (including reasoning summaries and all tool calls)
is saved as training data for fine-tuning Qwen3-4B.

## Architecture
```
User prompt → GPT-5.4 (Responses API)
                ├── reasoning item (summary captured)
                └── function_call item → execute tool → function_call_output → loop
                                                                                 ↓
                                                                         final message
```

## Setup

In [1]:
import json
import os
import sys
from pathlib import Path

from openai import AsyncOpenAI
from dotenv import load_dotenv

# Add nb/ to path so we can import bbb as a package
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "nb"))

from bbb.tools import TOOL_SCHEMAS, TOOL_FUNCTIONS
from bbb.agent import run_tool_calling_agent
from bbb.helpers__data_gen import SYSTEM_PROMPT, serialize_input_list

load_dotenv(PROJECT_ROOT / ".env")

client = AsyncOpenAI(base_url="https://us.api.openai.com/v1")

MODEL = "gpt-5.4"

## System Prompt

Uses the `developer` role (Responses API equivalent of `system`).
Instructs the model to use tools for research and produce a structured memo.
Reasoning happens via the model's internal reasoning tokens — no fake `<think>` tags needed.

In [2]:
# System prompt is imported from helpers__data_gen.py (shared across all BBB notebooks)
print(SYSTEM_PROMPT[:200] + "...")

You are a sell-side equity research analyst producing a brief research snapshot.

## Instructions
- Use the available tools to gather the data you need. Be efficient — call only what's necessary.
- Th...


## Tool Schemas

These are defined in `tools/stock_tools.py` and shared across all BBB notebooks.

In [3]:
# Quick look at what tools the agent has access to
# Responses API uses flat format: name/description/parameters at top level
for tool in TOOL_SCHEMAS:
    params = list(tool["parameters"]["properties"].keys())
    print(f"  {tool['name']}({', '.join(params)})")
    print(f"    → {tool['description']}\n")

  get_stock_news(ticker)
    → Get the 5 most recent news articles for a stock ticker.

  get_financials(ticker, statement_type, period)
    → Get financial statements (income, balance_sheet, or cashflow) for a stock.

  get_price_history(ticker, period, interval)
    → Get historical stock price data with summary statistics.

  get_recommendations(ticker, months_back)
    → Get analyst recommendations and recent upgrades/downgrades for a stock.



## Run the Agent

Uses `client.responses.create()` with reasoning enabled.
The agent loop handles:
- Parsing `function_call` items from `response.output`
- Executing tools and sending `function_call_output` back
- Passing reasoning items between turns (required for reasoning models)
- Capturing reasoning summaries for inspection

In [4]:
result = await run_tool_calling_agent(
    client=client,
    model=MODEL,
    user_prompt="Research NVDA focusing on financial health and growth potential.",
    system_prompt=SYSTEM_PROMPT,
    reasoning_effort="medium",
)

print(f"\nTotal input items: {len(result['input'])}")
print(f"Reasoning summaries: {len(result['reasoning_summaries'])}")
print(f"Tokens — input: {result['usage']['input_tokens']}, output: {result['usage']['output_tokens']}, reasoning: {result['usage']['reasoning_tokens']}")

  [1] Reasoning: **Researching financials for NVDA**

I need to gather a research snapshot on NVDA and use the appropriate tools for this...
  [1] Reasoning: **Gathering EPS and financial data**

I need to look into the EPS, focusing on both quarterly and annual figures, possib...
  [1] Called get_financials(ticker='NVDA', statement_type='income', period='quarterly')
  [1] Called get_financials(ticker='NVDA', statement_type='balance_sheet', period='quarterly')
  [1] Called get_price_history(ticker='NVDA', period='1y', interval='1wk')
  [1] Called get_recommendations(ticker='NVDA', months_back=12)
  [1] Called get_stock_news(ticker='NVDA')
  [2] Reasoning: **Clarifying company details**

I’m focusing on generating a financial snapshot to assess health and growth potential. T...
  [2] Reasoning: **Calculating financial metrics**

I need to rely strictly on the tools while identifying by ticker, which is fine. For ...
  [2] Reasoning: **Considering financial data**

I’m looking at the fin

### Inspect the full trajectory

Everything in order: developer prompt → user → reasoning → tool calls → tool results → ... → final memo.

In [5]:
# Print the full trajectory: input items + final output items, all in order
all_items = list(result["input"]) + list(result["output"])

for i, item in enumerate(all_items):
    # Dict items (our input messages and function_call_outputs)
    if isinstance(item, dict):
        role = item.get("role", item.get("type", "?"))
        content = item.get("content", item.get("output", ""))
        if role == "developer":
            print(f"[{i}] DEVELOPER\n{str(content)[:200]}...\n")
        elif role == "user":
            print(f"[{i}] USER\n{content}\n")
        elif item.get("type") == "function_call_output":
            print(f"[{i}] TOOL RESULT (call_id: {item.get('call_id', '?')[:20]})\n{str(content)[:200]}...\n")
        else:
            print(f"[{i}] {role.upper()}\n{str(content)[:200]}...\n")
    # SDK response objects
    else:
        item_type = getattr(item, "type", "unknown")
        if item_type == "reasoning":
            summary = "(no summary)"
            if getattr(item, "summary", None) and item.summary:
                summary = item.summary[0].text
            print(f"[{i}] REASONING\n{summary}\n")
        elif item_type == "function_call":
            print(f"[{i}] FUNCTION CALL: {item.name}\n  args: {item.arguments}\n")
        elif item_type == "message":
            text = ""
            if hasattr(item, "content") and item.content:
                text = item.content[0].text
            print(f"[{i}] ASSISTANT (final)\n{text[:500]}...\n")
        else:
            print(f"[{i}] {item_type.upper()}\n{str(item)[:200]}\n")

[0] DEVELOPER
You are a sell-side equity research analyst producing a brief research snapshot.

## Instructions
- Use the available tools to gather the data you need. Be efficient — call only what's necessary.
- Th...

[1] USER
Research NVDA focusing on financial health and growth potential.

[2] REASONING
**Researching financials for NVDA**

I need to gather a research snapshot on NVDA and use the appropriate tools for this. This includes financial income, balance sheet data, price history, news, and recommendations. I'm particularly interested in market cap and sector information. While I can't use a profile tool, I might infer the sector from stock news. I think I’ll fetch outputs regarding price history, income statements, and balance sheets while ensuring I don’t fabricate any missing data. Let's inspect the tool outputs!

[3] FUNCTION CALL: get_financials
  args: {"ticker":"NVDA","statement_type":"income","period":"quarterly"}

[4] FUNCTION CALL: get_financials
  args: {"ticker":

In [6]:
from IPython.display import Markdown, display

if result["output_text"]:
    display(Markdown(result["output_text"]))
else:
    print("No final text output — agent may have hit max iterations")

**Equity Research Snapshot**

**NVIDIA (NVDA)** | Semiconductors / AI | **~$4.10T market cap**  
*(Market cap derived from latest share count of 24.304B and last weekly close of $168.72.)*

**Key Metrics:**  
- **Revenue (TTM):** **$215.9B**  
- **Diluted EPS (TTM):** **$4.90**  
- **P/E:** **~34.4x** (using $168.72 / $4.90)  
- **Latest quarter revenue:** **$68.1B**, **+73% YoY** vs. $39.3B  
- **Latest quarter diluted EPS:** **$1.76**, vs. **$0.89** YoY  
- **Gross margin:** **75.0%**; **Operating margin:** **65.0%**; **Net margin:** **63.1%**  
- **Debt / Equity:** **0.07x**  
- **Cash + short-term investments:** **$62.6B** vs. **total debt $11.0B**  
- **Current ratio:** **~3.9x**

**Recent Developments:**  
- Latest reported quarter was another step-up in scale: revenue rose to **$68.1B** from **$57.0B** the prior quarter and **$39.3B** a year earlier.  
- Analyst sentiment remains very constructive: recent target hikes include **Raymond James to $323** (from $291), **Wedbush to $300** (from $230), **Citi to $300** (from $270), and **BofA to $300** (from $275).  
- Available news feed was light on company-specific operating updates; most actionable recent data came from earnings and analyst revisions.

**Financial Highlights:**  
- Balance sheet is exceptionally strong: **$157.3B** equity, **$93.4B** working capital, and substantial net cash.  
- Profitability remains elite even at scale, with latest-quarter **gross profit of $51.1B** and **operating income of $44.3B**.  
- Growth is still accelerating in absolute dollars, but working capital is expanding too: **inventory rose to $21.4B** from **$10.1B** a year earlier, and **receivables increased to $38.5B** from **$23.1B**.

**Price Action:**  
- **Last price:** **$168.72**  
- **52-week range:** **$94.29 - $202.47**  
- **1-year change:** **+53.9%**  
- Shares rallied sharply off the April low, peaked near **$202.47** in late October, and have since consolidated; the stock is now **~16.7% below** the 52-week high.

**Analyst Consensus:**  
- Ratings mix: **57 Buy/Strong Buy, 2 Hold, 1 Sell**  
- The tool did not provide a formal consensus target; across recent March price-target updates, targets ranged from **$235 to $360**, averaging **~$286**.

**Bull Case:**  
- Explosive top-line growth continues: latest-quarter revenue was **+73% YoY**, with sequential growth also strong.  
- Financial health is outstanding, with **low leverage**, **ample liquidity**, and **very high margins**.  
- Analyst positioning remains overwhelmingly positive, with multiple upward target revisions after recent results.

**Bear Case:**  
- Even after the pullback, valuation is still elevated at **~34x TTM EPS**, leaving less room for execution misses.  
- Rapid balance-sheet build in **inventory** and **receivables** could become a watchpoint if demand normalizes.  
- Stock volatility remains high; despite strong fundamentals, shares have corrected materially from peak levels.

**Bottom Line:**** NVIDIA’s financial health is best-in-class and growth remains extraordinary, but the key debate is no longer balance-sheet risk — it is how long current AI-driven demand can sustain margins and valuation at this scale.

## Save the Trajectory

We serialize the input list for training data. Responses API items
that are SDK objects get serialized via their `.model_dump()` method.

In [7]:
# serialize_input_list is imported from helpers__data_gen.py

output_dir = PROJECT_ROOT / "data" / "bbb"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "tool_calling_trajectories.jsonl"

record = {
    "input": serialize_input_list(result["input"]),
    "output": serialize_input_list(result["output"]),
    "tools": TOOL_SCHEMAS,
    "reasoning_summaries": result["reasoning_summaries"],
    "usage": result["usage"],
}

with open(output_file, "a") as f:
    f.write(json.dumps(record) + "\n")

print(f"Saved trajectory to {output_file}")

Saved trajectory to /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/tool_calling_trajectories.jsonl


## Test with a Few More Tickers

In [8]:
from tqdm.asyncio import tqdm_asyncio

test_tickers = [
    ("AAPL", "competitive position and market share"),
    ("TSLA", "recent news and analyst sentiment"),
    ("JPM", "financial health and balance sheet strength"),
]

async def process_ticker(ticker, focus):
    print(f"\n{'='*60}")
    print(f"Researching {ticker} — {focus}")
    print(f"{'='*60}")

    res = await run_tool_calling_agent(
        client=client,
        model=MODEL,
        user_prompt=f"Research {ticker} focusing on {focus}.",
        system_prompt=SYSTEM_PROMPT,
        reasoning_effort="medium",
    )

    # Count tool calls from the input list
    n_tool_results = sum(
        1 for item in res["input"]
        if isinstance(item, dict) and item.get("type") == "function_call_output"
    )
    memo_len = len(res["output_text"]) if res["output_text"] else 0
    print(f"  -> {n_tool_results} tool calls, {len(res['reasoning_summaries'])} reasoning steps, memo: {memo_len} chars")
    print(f"  -> Tokens: {res['usage']['input_tokens']} in / {res['usage']['output_tokens']} out / {res['usage']['reasoning_tokens']} reasoning")

    # Prepare record for writing after gather()
    record = {
        "input": serialize_input_list(res["input"]),
        "output": serialize_input_list(res["output"]),
        "tools": TOOL_SCHEMAS,
        "reasoning_summaries": res["reasoning_summaries"],
        "usage": res["usage"],
    }
    return record

# Create tasks for all tickers
tasks = [process_ticker(ticker, focus) for ticker, focus in test_tickers]
results = await tqdm_asyncio.gather(*tasks, desc="Generating trajectories")

# Write all results at the end
with open(output_file, "a") as f:
    for record in results:
        f.write(json.dumps(record) + "\n")

Generating trajectories:   0%|          | 0/3 [00:00<?, ?it/s]


Researching TSLA — recent news and analyst sentiment

Researching AAPL — competitive position and market share

Researching JPM — financial health and balance sheet strength
  [1] Reasoning: **Preparing equity research snapshot**

I need to create an equity research snapshot for AAPL, focusing on its competiti...
  [1] Reasoning: **Gathering equity snapshot**

I need to produce an equity snapshot for JPMorgan Chase with the ticker JPM, focusing on ...
  [1] Called get_financials(ticker='AAPL', statement_type='income', period='annual')
  [1] Called get_financials(ticker='AAPL', statement_type='balance_sheet', period='annual')
  [1] Called get_price_history(ticker='AAPL', period='1y', interval='1wk')
  [1] Called get_recommendations(ticker='AAPL', months_back=12)
  [1] Called get_financials(ticker='JPM', statement_type='balance_sheet', period='quarterly')
  [1] Called get_financials(ticker='JPM', statement_type='income', period='quarterly')
  [1] Called get_stock_news(ticker='AAPL')
  [

Generating trajectories:  33%|███▎      | 1/3 [00:55<01:51, 55.53s/it]

  [2] Reasoning: **Calculating financial metrics**

I could use the latest annual financials ending September 30, 2025, and label it as t...
  [2] Reasoning: **Discussing target price estimates**

I can provide a recent target price range and possibly calculate an average from ...
  [2] Reasoning: **Gathering stock insights**

I need to pull recent news and analyst insights. For instance, Wedbush maintained an Outpe...
  [2] Reasoning: **Calculating financial metrics**

I’m computing margins: gross margin is about 46.9%, operating margin is around 32.0%,...
  [2] Agent finished — produced final response
  -> 5 tool calls, 5 reasoning steps, memo: 3802 chars
  -> Tokens: 15276 in / 2461 out / 1484 reasoning


Generating trajectories:  67%|██████▋   | 2/3 [01:10<00:31, 31.71s/it]

  [2] Reasoning: **Calculating market cap and sector**

I need to find the market cap and sector information, and I’m thinking maybe pric...
  [2] Reasoning: **Finding analyst consensus target price**

I need the analyst consensus target price, but the recommendations tool does...
  [2] Reasoning: **Summarizing recent analyst news**

I’m calculating the total analyst count, which seems to be 48. I need to summarize ...
  [2] Reasoning: **Analyzing financial details**

I'm looking into China's investor focus on the AI narrative, particularly with Optimus....
  [2] Reasoning: **Examining recent closes**

I've analyzed the last six weekly closes: February 16 at 411.82, February 23 at 402.51, Mar...
  [2] Reasoning: **Calculating market cap and targets**

I’m trying to determine which shares to use for market cap. Using ordinary share...
  [2] Reasoning: **Sorting and finding median**

I’m thinking the median might be around 417.5. I should quickly sort these values: 25.28...
  [2] Reasoni

Generating trajectories: 100%|██████████| 3/3 [01:16<00:00, 25.58s/it]

  [2] Reasoning: **Calculating financial metrics**

I need to compute some key financial metrics. The TTM revenue for the last four quart...
  [2] Reasoning: **Assessing balance sheet strength**

I'm looking to evaluate balance sheet strength through equity and tangible book va...
  [2] Reasoning: **Reviewing financial metrics**

I should detail the 4Q25 revenue at 45.8 billion, with an EPS of 4.63. The common equit...
  [2] Reasoning: **Analyzing price trends**

I want to summarize the price action trend, noting it's up 19.8% over the past year. It peak...
  [2] Reasoning: **Analyzing financial data**

I’m calculating some financial figures: starting with a total that sums to 72.595 billion....
  [2] Reasoning: **Discussing balance sheet insights**

Since I'm focusing on balance sheet strength, I need to highlight the tool limita...
  [2] Agent finished — produced final response
  -> 5 tool calls, 7 reasoning steps, memo: 2941 chars
  -> Tokens: 14722 in / 2746 out / 1938 reasoning
